Greenhouse gas report lab notebook for Document AI, with markdown aligned to one 2024 report and the regional CO₂ figure on page 2.
*Co-authored with CoCo*

# Greenhouse Gas Report: Regional CO₂ with Cortex Search

*An end-to-end workflow with AI_PARSE_DOCUMENT, Cortex Search, and AI_COMPLETE*

This notebook parses one synthetic greenhouse gas report in AGENTIC mode, indexes its page text, visual summaries, and chart details in Cortex Search, and answers a question about the regional CO₂ figure on page 2 with `claude-sonnet-5-5`.

Upload `greenhouse_gas_report_2024.pdf` to the `GHG_REPORTS` stage before running the parse cells.

In [ ]:
%%sql -r ghg_setup
CREATE DATABASE IF NOT EXISTS VHOL_DOCUMENT_AI_DB;
CREATE SCHEMA IF NOT EXISTS VHOL_DOCUMENT_AI_DB.DOCUMENT_AI;

USE DATABASE VHOL_DOCUMENT_AI_DB;
USE SCHEMA DOCUMENT_AI;

SELECT CURRENT_ROLE(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE();

In [ ]:
%%sql -r ghg_stage
CREATE STAGE IF NOT EXISTS GHG_REPORTS
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
    DIRECTORY = (ENABLE = TRUE);

### Upload and Check the PDF

1. Download [greenhouse_gas_report_2024.pdf](https://github.com/Snowflake-Labs/sfquickstarts/blob/master/site/sfguides/src/document-intelligence-ai-functions/assets/greenhouse_gas_report_2024.pdf). For GitHub-hosted files, select **Download raw file**.
2. In Snowsight, go to **Ingestion > Add Data > Load files into a Stage** and upload the PDF to `GHG_REPORTS`.
3. Run the next cell to refresh the directory table and confirm the file is listed.

In [ ]:
%%sql -r ghg_refresh
ALTER STAGE GHG_REPORTS REFRESH;

LIST @GHG_REPORTS;

## Register the Report

The report year is read from the filename. The query should return one row for `greenhouse_gas_report_2024.pdf` with report year 2024.

In [ ]:
%%sql -r ghg_files
CREATE OR REPLACE TEMP TABLE GHG_FILES AS
SELECT
    relative_path,
    SPLIT_PART(relative_path, '/', -1) AS filename,
    REGEXP_SUBSTR(SPLIT_PART(relative_path, '/', -1), '(20[0-9]{2})', 1, 1, 'e', 1) AS report_year,
    size AS file_size_bytes,
    last_modified
FROM DIRECTORY(@GHG_REPORTS)
WHERE relative_path ILIKE '%.pdf';

SELECT * FROM GHG_FILES ORDER BY report_year;

## Parse the Report with AGENTIC Mode
Duration: 10 minutes

`AI_PARSE_DOCUMENT` in AGENTIC mode returns page text and a visual interpretation for each image. `page_split` returns one record per page. The final `TRUE` returns error details in the result instead of failing the query.

This step calls the model and can take a few minutes.

In [ ]:
%%sql -r ghg_parse
CREATE OR REPLACE TEMP TABLE GHG_PARSED AS
SELECT
    relative_path,
    filename,
    report_year,
    AI_PARSE_DOCUMENT(
        TO_FILE('@GHG_REPORTS', relative_path),
        {'mode': 'AGENTIC', 'page_split': TRUE},
        TRUE
    ) AS raw_parse
FROM GHG_FILES;

SELECT
    filename,
    report_year,
    ARRAY_SIZE(raw_parse:value:pages) AS returned_pages,
    raw_parse:error AS call_error,
    raw_parse:value:error AS document_error
FROM GHG_PARSED
ORDER BY report_year;

## Split Pages, Visual Summaries, and Chart Details

The parsed output is split into one row per page and one row per image. Images that return an error are skipped. Each image's structured details are kept so the chart series can be indexed.

In [ ]:
%%sql -r ghg_pages
CREATE OR REPLACE TEMP TABLE GHG_PAGES AS
SELECT
    r.relative_path,
    r.filename,
    r.report_year,
    p.value:index::NUMBER + 1 AS pdf_page_number,
    p.value:content::VARCHAR AS page_content,
    p.value:images AS images
FROM GHG_PARSED AS r,
     LATERAL FLATTEN(INPUT => r.raw_parse:value:pages) AS p
WHERE NULLIF(r.raw_parse:error::VARCHAR, 'null') IS NULL;

CREATE OR REPLACE TEMP TABLE GHG_VISUALS AS
SELECT
    p.relative_path,
    p.filename,
    p.report_year,
    p.pdf_page_number,
    i.index AS image_index,
    i.value:results:image_type::VARCHAR AS image_type,
    i.value:results:summary::VARCHAR AS summary,
    i.value:results:details AS details
FROM GHG_PAGES AS p,
     LATERAL FLATTEN(INPUT => p.images) AS i
WHERE NULLIF(TRIM(i.value:results:summary::VARCHAR), '') IS NOT NULL
  AND COALESCE(LOWER(i.value:results:description::VARCHAR), '') <> 'error';

SELECT report_year, COUNT(*) AS pages FROM GHG_PAGES GROUP BY report_year ORDER BY report_year;

SELECT report_year, COUNT(*) AS visuals_with_summary FROM GHG_VISUALS GROUP BY report_year ORDER BY report_year;

## List All Extracted Images

This lists every image the parse returned, with its page, image ID, type, summary, and structured details. Images without a summary or details show NULL in those columns.

In [ ]:
%%sql -r ghg_all_images
SELECT
    p.pdf_page_number,
    i.index AS image_index,
    i.value:id::VARCHAR AS image_id,
    i.value:results:image_type::VARCHAR AS image_type,
    i.value:results:summary::VARCHAR AS summary,
    i.value:results:details AS details
FROM GHG_PAGES AS p,
     LATERAL FLATTEN(INPUT => p.images) AS i
ORDER BY p.pdf_page_number, i.index;

## Build the Search Content

Index three kinds of content: the page text, the visual summaries, and the chart details. Each chunk repeats the report year, PDF page, and content type, so a search can be traced back to its page.

In [ ]:
%%sql -r ghg_chunks
CREATE OR REPLACE TEMP TABLE GHG_SEARCH_UNITS AS
SELECT relative_path, filename, report_year, pdf_page_number,
       'text'::VARCHAR AS content_type,
       page_content AS body
FROM GHG_PAGES
WHERE NULLIF(TRIM(page_content), '') IS NOT NULL
UNION ALL
SELECT relative_path, filename, report_year, pdf_page_number,
       'visual_summary'::VARCHAR AS content_type,
       summary AS body
FROM GHG_VISUALS
UNION ALL
SELECT
    v.relative_path,
    v.filename,
    v.report_year,
    v.pdf_page_number,
    'visual_details'::VARCHAR AS content_type,
    LISTAGG(d.path || ': ' || d.value::VARCHAR, '\n') WITHIN GROUP (ORDER BY d.path) AS body
FROM GHG_VISUALS AS v,
     LATERAL FLATTEN(INPUT => v.details, RECURSIVE => TRUE) AS d
WHERE TYPEOF(d.value) NOT IN ('OBJECT', 'ARRAY', 'NULL_VALUE')
  AND d.path NOT LIKE '%value_range%'
  AND d.path NOT LIKE '%value_precision%'
GROUP BY v.relative_path, v.filename, v.report_year, v.pdf_page_number, v.image_index;

CREATE OR REPLACE TABLE GHG_SEARCH_CHUNKS
    DATA_RETENTION_TIME_IN_DAYS = 1 AS
SELECT
    MD5(u.relative_path || '|' || u.pdf_page_number::VARCHAR || '|' ||
        u.content_type || '|' || c.index::VARCHAR) AS chunk_id,
    u.relative_path,
    u.filename,
    u.report_year,
    u.pdf_page_number,
    u.content_type,
    'Report: Greenhouse Gas Report ' || u.report_year ||
    '. PDF page: ' || u.pdf_page_number::VARCHAR ||
    '. Content: ' || u.content_type || CHR(10) || c.value::VARCHAR AS search_text
FROM GHG_SEARCH_UNITS AS u,
     LATERAL FLATTEN(INPUT => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
         u.body, 'markdown', 1400, 150
     )) AS c
WHERE NULLIF(TRIM(c.value::VARCHAR), '') IS NOT NULL;

ALTER TABLE GHG_SEARCH_CHUNKS SET CHANGE_TRACKING = TRUE;

SELECT report_year, content_type, COUNT(*) AS chunks
FROM GHG_SEARCH_CHUNKS
GROUP BY report_year, content_type
ORDER BY report_year, content_type;

## Create the Cortex Search Service

The service uses the warehouse selected in the notebook and filters on report year, content type, and filename. The index builds after creation, which can take a short time. If the retrieval below returns nothing right away, wait a minute and run the DESCRIBE statement again to check that `indexing_state` is `ACTIVE`.

In [ ]:
%%sql -r ghg_search_service
-- Replaces only the Search service created by this notebook.
DROP CORTEX SEARCH SERVICE IF EXISTS GHG_REPORT_SEARCH;

EXECUTE IMMEDIATE $$
DECLARE
    wh VARCHAR DEFAULT CURRENT_WAREHOUSE();
    ddl VARCHAR;
    missing_warehouse EXCEPTION (-20001, 'Select a lab warehouse before creating Search.');
BEGIN
    IF (wh IS NULL) THEN
        RAISE missing_warehouse;
    END IF;
    ddl := 'CREATE OR REPLACE CORTEX SEARCH SERVICE GHG_REPORT_SEARCH '
        || 'ON search_text ATTRIBUTES report_year, content_type, filename '
        || 'WAREHOUSE = "' || REPLACE(wh, '"', '""') || '" '
        || 'TARGET_LAG = ''1 hour'' INITIALIZE = ON_CREATE '
        || 'AS SELECT chunk_id, search_text, relative_path, filename, '
        || 'report_year, pdf_page_number, content_type FROM GHG_SEARCH_CHUNKS';
    EXECUTE IMMEDIATE :ddl;
    RETURN 'Created GHG_REPORT_SEARCH.';
END;
$$;

DESCRIBE CORTEX SEARCH SERVICE GHG_REPORT_SEARCH;

## Retrieve Passages for the Question

The question is searched within the 2024 report. Each passage is shown with the chart details from the same page, so the regional figure values reach the answer step.

In [ ]:
%%sql -r ghg_hits
CREATE OR REPLACE TEMP TABLE GHG_RETRIEVED_PASSAGES AS
WITH hits AS (
    SELECT
        r.value:FILENAME::VARCHAR AS filename,
        r.value:REPORT_YEAR::VARCHAR AS report_year,
        r.value:PDF_PAGE_NUMBER::NUMBER AS pdf_page_number,
        r.value:CONTENT_TYPE::VARCHAR AS content_type,
        r.value:SEARCH_TEXT::VARCHAR AS passage
    FROM (
        SELECT PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'GHG_REPORT_SEARCH',
            '{"query": "Which year had the highest and which had the lowest annual regional total CO2 emissions between 2020 and 2023", "columns": ["SEARCH_TEXT", "FILENAME", "REPORT_YEAR", "PDF_PAGE_NUMBER", "CONTENT_TYPE"], "filter": {"@eq": {"REPORT_YEAR": "2024"}}, "limit": 6}'
        )) AS response
    ) AS s,
    LATERAL FLATTEN(INPUT => s.response:results) AS r
),
details AS (
    SELECT report_year, pdf_page_number, LISTAGG(search_text, '\n') WITHIN GROUP (ORDER BY chunk_id) AS chart_details
    FROM GHG_SEARCH_CHUNKS
    WHERE content_type = 'visual_details'
    GROUP BY report_year, pdf_page_number
)
SELECT h.filename, h.report_year, h.pdf_page_number, h.content_type, h.passage, d.chart_details
FROM hits AS h
LEFT JOIN details AS d
  ON d.report_year = h.report_year
 AND d.pdf_page_number = h.pdf_page_number
ORDER BY h.pdf_page_number;

SELECT * FROM GHG_RETRIEVED_PASSAGES ORDER BY pdf_page_number;

## Answer the Question with AI_COMPLETE

The answer step passes the retrieved passages and chart details to `claude-sonnet-5-5`. The model names the year with the highest and lowest annual regional total CO₂ between 2020 and 2023, using values read from the chart where the passages give them. It cites each claim by report year, PDF page, and source type (text, visual summary, or visual details), and says when a value comes from a chart description rather than printed text.

In [ ]:
WITH context AS (
    SELECT
        LISTAGG(
            'Report ' || report_year || ', PDF page ' || pdf_page_number::VARCHAR || ' (' || content_type || '):\n' || passage ||
            IFF(chart_details IS NULL, '', '\nChart details for page ' || pdf_page_number::VARCHAR || ':\n' || chart_details),
            '\n---\n'
        ) WITHIN GROUP (ORDER BY pdf_page_number) AS passages
    FROM GHG_RETRIEVED_PASSAGES
)
SELECT AI_COMPLETE(
    'claude-sonnet-5-5',
    'You answer questions about a 2024 greenhouse gas report using only the passages below. Question: which year had the highest and which had the lowest annual regional total CO2 emissions between 2020 and 2023? Cite each claim as [Report 2024, PDF page P, source], where source is the content label of the passage you used: text, visual_summary, or visual_details. If the passages do not name a region or the fire series, say so rather than guessing. If the passages do not contain the answer, say what is missing.\n\nPassages:\n' || passages
) AS answer
FROM context;

## Cleanup

This cell drops `VHOL_DOCUMENT_AI_DB`, which removes the stage and uploaded PDF, the Cortex Search service, and the tables created in that database.

Do not run this against a shared event database. In a shared account, ask the host which objects you can remove.

In [ ]:
%%sql -r ghg_cleanup
DROP DATABASE IF EXISTS VHOL_DOCUMENT_AI_DB;

## Conclusion

You parsed a greenhouse gas report with AGENTIC mode, indexed its page text, visual summaries, and chart details, and asked a question that needed both printed text and chart values. The answer gives approximate regional values and cites each claim by report year, PDF page, and source type.